# Daily Challenge: Bank Customer Churn Prediction

**Course:** Developers Institute  **Week 5 - Day 3**  
**Author:** Alex Goldbaum

Goal: predict whether a bank customer will **leave the bank** (churn) given
their demographics, account information and activity. We use Logistic Regression
with proper preprocessing (encoding + scaling), evaluate with class-aware metrics
(since churners are the minority ~20%), and translate the results into
retention recommendations.

Dataset: Kaggle **Bank Customer Churn Modelling** (10,000 customers).
The notebook auto-generates a realistic synthetic version with the same schema
if the CSV is not present.


## Setup


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load the data

If `Churn_Modelling.csv` is in the working directory we use it. Otherwise we
generate a synthetic dataset with the same columns (and realistic correlations).


In [ ]:
import os

CSV_PATH = 'Churn_Modelling.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded real data from {CSV_PATH}')
else:
    print('CSV not found - generating realistic synthetic churn data (10,000 rows).')
    n = 10_000
    rng = np.random.default_rng(RANDOM_STATE)

    geography = rng.choice(['France', 'Germany', 'Spain'], size=n, p=[0.5, 0.25, 0.25])
    gender    = rng.choice(['Female', 'Male'], size=n, p=[0.46, 0.54])
    age       = np.clip(rng.normal(39, 10, n).round().astype(int), 18, 92)
    tenure    = rng.integers(0, 11, size=n)
    credit    = np.clip(rng.normal(650, 95, n).round().astype(int), 350, 850)
    balance   = np.where(rng.random(n) < 0.36, 0.0,
                         np.clip(rng.normal(120_000, 60_000, n), 0, 250_000)).round(2)
    num_prod  = rng.choice([1, 2, 3, 4], size=n, p=[0.51, 0.46, 0.026, 0.004])
    has_card  = rng.choice([0, 1], size=n, p=[0.30, 0.70])
    is_active = rng.choice([0, 1], size=n, p=[0.49, 0.51])
    salary    = rng.uniform(11.58, 199_992.48, size=n).round(2)

    # Build a churn probability that mirrors the real-world drivers
    risk = (
        0.85 * ((age - 39) / 10)              # older customers churn more
      - 1.20 * (is_active - 0.5)              # inactivity is the strongest driver
      + 0.95 * (geography == 'Germany').astype(float)
      + 2.40 * (num_prod >= 3).astype(float)  # 3+ products = strong red flag
      - 0.55 * (num_prod == 2).astype(float)  # 2 products is the sweet spot
      + 0.40 * (balance > 100_000).astype(float)
      + 0.15 * (gender == 'Female').astype(float)
    )
    probs = 1 / (1 + np.exp(-(risk - 1.6)))   # shift to target ~20% positives
    exited = (rng.random(n) < probs).astype(int)

    df = pd.DataFrame({
        'RowNumber': np.arange(1, n + 1),
        'CustomerId': rng.integers(15_000_000, 16_000_000, size=n),
        'Surname': ['Customer_%05d' % i for i in range(n)],
        'CreditScore': credit,
        'Geography': geography,
        'Gender': gender,
        'Age': age,
        'Tenure': tenure,
        'Balance': balance,
        'NumOfProducts': num_prod,
        'HasCrCard': has_card,
        'IsActiveMember': is_active,
        'EstimatedSalary': salary,
        'Exited': exited,
    })

print(f'Shape: {df.shape}')
df.head()


## 2. Exploratory Data Analysis


In [ ]:
df.info()


In [ ]:
# Missing values
missing = df.isna().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else '  (no missing values)')


In [ ]:
# Class balance
churn_counts = df['Exited'].value_counts().rename({0: 'Retained', 1: 'Churned'})
churn_rate = df['Exited'].mean() * 100
print(churn_counts)
print(f'\nChurn rate: {churn_rate:.2f}%')

plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=df, palette=['steelblue', 'tomato'])
plt.xticks([0, 1], ['Retained', 'Churned'])
plt.title(f'Class distribution (churn rate = {churn_rate:.1f}%)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Numeric statistics
df.describe().round(2)


In [ ]:
# Churn rate by categorical / discrete features
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, col in zip(axes.flat, ['Geography', 'Gender', 'NumOfProducts', 'IsActiveMember']):
    rate = df.groupby(col)['Exited'].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index.astype(str), y=rate.values, palette='magma', ax=ax)
    ax.set_title(f'Churn rate by {col}', fontweight='bold')
    ax.set_ylabel('Churn rate (%)')
    ax.set_ylim(0, max(60, rate.max() + 5))
    for i, v in enumerate(rate.values):
        ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Numeric features vs churn (distributions)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age', 'Balance', 'CreditScore']):
    sns.histplot(data=df, x=col, hue='Exited', kde=True, bins=30, ax=ax,
                 palette=['steelblue', 'tomato'])
    ax.set_title(f'{col} distribution by churn', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Numeric correlation heatmap (excluding identifiers / strings)
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
            'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
corr = df[num_cols].corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation matrix — numeric features', fontweight='bold')
plt.tight_layout()
plt.show()


**EDA findings.**
- The classes are imbalanced (~20% churners) — accuracy alone will be misleading.
- Geography matters: German customers churn at a much higher rate than French/Spanish.
- `NumOfProducts` has a non-monotonic effect: customers with 1 product churn often,
  customers with 2 churn the least, customers with 3 or 4 churn the most
  (likely sign of over-selling / dissatisfaction).
- Inactive members churn much more than active members — actionable signal.
- Older customers churn more on average; `Age` is the strongest individual numeric
  predictor.


## 3. Data Preprocessing

Drop the identifier columns (`RowNumber`, `CustomerId`, `Surname` — no
predictive signal), one-hot encode `Geography` and `Gender`, scale the numeric
columns with `StandardScaler`. The scaler is fitted **only on the training fold**
to avoid leakage. All this happens inside a `Pipeline` so the same transform is
applied to the test set exactly the same way.


In [ ]:
drop_cols = ['RowNumber', 'CustomerId', 'Surname']
X = df.drop(columns=drop_cols + ['Exited'])
y = df['Exited']

categorical = ['Geography', 'Gender']
numeric = [c for c in X.columns if c not in categorical]

print('Categorical features :', categorical)
print('Numeric features     :', numeric)


In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical),
])


In [ ]:
# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} rows ({y_train.mean()*100:.2f}% churned)')
print(f'Test : {X_test.shape[0]} rows ({y_test.mean()*100:.2f}% churned)')


## 4. Build and train the model

Logistic Regression with `class_weight='balanced'` so the loss weighs churners
and non-churners equally (the right call for an imbalanced retention problem,
where missing a churner is more costly than a false alarm).


In [ ]:
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE,
    )),
])

model.fit(X_train, y_train)
print('Model trained.')


## 5. Evaluate the model


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Test set metrics ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1-score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix — test set', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Churn prediction', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Cross-validated AUC (more honest than single test split)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-fold CV ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})')


## 6. Feature importance

Standardized logistic regression coefficients translate to log-odds change per
1-standard-deviation increase in the feature (or per category vs the dropped
reference). Bigger |coefficient| ⇒ stronger effect on churn probability.


In [ ]:
# Pull out the names after preprocessing
ohe = model.named_steps['preprocess'].named_transformers_['cat']
ohe_names = ohe.get_feature_names_out(categorical)
feature_names = list(numeric) + list(ohe_names)

coefs = model.named_steps['clf'].coef_[0]
imp = pd.Series(coefs, index=feature_names).sort_values()

plt.figure(figsize=(9, 6))
colors = ['tomato' if v > 0 else 'steelblue' for v in imp.values]
plt.barh(imp.index, imp.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic regression coefficients (positive = increases churn risk)',
          fontweight='bold')
plt.tight_layout()
plt.show()

print('Coefficients ranked by |effect|:')
print(imp.reindex(imp.abs().sort_values(ascending=False).index).round(3))


## 7. Business insights and recommendations

**1. Who are the at-risk customers (from the coefficients)?**
- **Inactive members** (`IsActiveMember=0`) — biggest negative effect when set
  to 1, i.e. activity strongly *reduces* churn.
- **German customers** — `Geography_Germany` has a large positive coefficient
  (much higher base churn than France/Spain).
- **Older customers** — `Age` is a strong positive predictor.
- **Customers with 3+ products** — `NumOfProducts` positive coefficient
  (oversold customers leave; the sweet spot is 2 products).
- **Balance** has a moderate positive coefficient: customers with high idle
  balances may feel they are not getting value from the bank.

**2. Recommendations for the retention team.**
1. **Reactivation campaigns for inactive members.** The single highest-leverage
   action. Personalised offers, fee waivers, or a relationship banker call.
2. **German market deep-dive.** The base rate is much higher than other regions;
   investigate root causes (product fit, fees, service quality) — this is a
   strategic, not just a tactical, issue.
3. **Care for older customers.** Onboard them to digital channels with extra
   support, offer dedicated phone lines, and personalised products.
4. **Review the 3- and 4-product segment.** A high-product count predicting
   churn is a red flag — clients may be over-sold and frustrated.
5. **Tune the decision threshold to the cost of a missed churner.** If retaining
   one customer is worth far more than the cost of contacting an inactive one,
   lower the threshold below 0.5 to capture more true churners.

**3. Caveats.**
- This is a Logistic Regression baseline. A Gradient Boosting model would likely
  add 2–4 ROC-AUC points and capture non-linear interactions
  (e.g., `Age × IsActiveMember`).
- Model fairness: before deploying, audit churn predictions across `Geography`,
  `Gender` and age bands to make sure recommendations are not unintentionally
  discriminatory.
- Concept drift: re-train regularly. Customer behaviour shifts with the economy,
  product changes, and competitor moves.
